## Q1

In [2]:
import torch 
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import random

In [3]:
# loading the model & tokenizer

model_name = "roberta-base"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
# loading n-random sentences

random.seed(42)
n = 100000
with open("dataset.txt", "r", encoding = "utf-8") as f:
    all_lines = f.readlines()
sampled = random.sample(all_lines, n)

sentences = []
for line in sampled:
    line = line.strip()
    if line:
        sentences.append(line)
len(sentences)

89122

In [8]:
# generating contextual embeddings

token_occurrences = {}
for sentence in tqdm(sentences):
    encoded = tokenizer(sentence, return_tensors = "pt", truncation = True, max_length = 512).to(device)
    with torch.no_grad():
        output = model(**encoded)
    hidden = output.last_hidden_state.squeeze(0)
    input_ids = encoded["input_ids"].squeeze(0)
    for tok_id, emb in zip(input_ids.tolist(), hidden):
        tok_id = int(tok_id)
        if tok_id not in token_occurrences:
            token_occurrences[tok_id] = []
        token_occurrences[tok_id].append(emb.cpu())

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 89122/89122 [15:45<00:00, 94.21it/s]


In [10]:
# computing static embeddings

static_embeddings = {}
for tok_id, embs in token_occurrences.items():
    embs = torch.stack(embs)
    static_embeddings[tok_id] = embs.mean(dim = 0)
len(static_embeddings)

41967

I used a random sample of 100,000 sentences from the provided dataset to generate contextual and static embeddings. I implemented the required pipeline using the RoBERTa transformer encoder, tokenized all sampled sentences with the corresponding tokenizer, computed contextualized embeddings for every token, and then averaged all occurrences of each token to produce the final static embeddings. In total, I obtained static embeddings for 41,967 unique subword tokens in the vocabulary.

I spent around 5 hours on Problem 1, including sampling the data, coding the implementation, debugging long-sequence issues, running the model, and computing the averaged embeddings.

## Q2

In [14]:
# loading the glove vocabulary

max_words = 50000    
glove_words = []
with open("glove.6B.300d-vocabulary.txt", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= max_words:
            break
        w = line.strip()
        if w:
            glove_words.append(w)
len(glove_words)

50000

In [15]:
# creating word embeddings by averaging token embeddings

word_embeddings = {}
for word in glove_words:
    encoded = tokenizer(word, add_special_tokens=False)
    token_ids = encoded["input_ids"]
    collected = []
    for tid in token_ids:
        if tid in static_embeddings:
            collected.append(static_embeddings[tid])
    if len(collected) > 0:
        stacked = torch.stack(collected)
        word_embeddings[word] = stacked.mean(dim=0)
len(word_embeddings)

49344

In [16]:
# implementing the most_similar() function

import torch.nn.functional as F

def most_similar(query_word, word_embeddings, topn=10):
    if query_word not in word_embeddings:
        return f"'{query_word}' not found in the vocabulary."
    query_vec = word_embeddings[query_word]
    sims = {}
    for word, vec in word_embeddings.items():
        if word == query_word:
            continue
        sim = F.cosine_similarity(query_vec, vec, dim=0).item()
        sims[word] = sim
    return sorted(sims.items(), key=lambda x: x[1], reverse=True)[:topn]

In [20]:
# example - "cactus"
most_similar("cactus", word_embeddings)

[('calcavecchia', 0.9831995368003845),
 ('caucasus', 0.9824233651161194),
 ('curtis', 0.9823352694511414),
 ('cpa', 0.9822880029678345),
 ('cops', 0.9821260571479797),
 ('cassius', 0.9818991422653198),
 ('carcasses', 0.9818729162216187),
 ('carcinoma', 0.9817315936088562),
 ('carpets', 0.9815269112586975),
 ('cistercian', 0.9814989566802979)]

In [21]:
# example - "cake"
most_similar("cake", word_embeddings)

[('fleetwood', 0.9485126733779907),
 ("''", 0.9452746510505676),
 ('heartburn', 0.9443551301956177),
 ('fastballs', 0.9420613646507263),
 ('farmington', 0.9419302940368652),
 ('headlines', 0.9416500329971313),
 ('johnstone', 0.9416301846504211),
 ('heartland', 0.9410415291786194),
 ('despairing', 0.9408101439476013),
 ('hillman', 0.9400108456611633)]

In [22]:
# example - "angry"
most_similar("angry", word_embeddings)

[('ssangyong', 0.983801543712616),
 ('barangay', 0.9837847352027893),
 ('hungry', 0.9831001162528992),
 ('pyeongchang', 0.9826205968856812),
 ('parry', 0.9825607538223267),
 ('merry', 0.9825559854507446),
 ('barry', 0.9824526309967041),
 ('ziyang', 0.9822881817817688),
 ('angie', 0.9821722507476807),
 ('gerry', 0.9817764759063721)]

In [23]:
# example - "quickly"
most_similar("quickly", word_embeddings)

[('clearly', 1.0),
 ('increasingly', 1.0),
 ('quietly', 1.0),
 ('surely', 1.0),
 ('tightly', 1.0),
 ('hugely', 1.0),
 ('popularly', 1.0),
 ('remotely', 1.0),
 ('solidly', 1.0),
 ('ly', 1.0)]

In [24]:
# example - "between"
most_similar("between", word_embeddings)

[('divorcing', 0.9247531890869141),
 ('blending', 0.9246541857719421),
 ('confrontations', 0.9242427945137024),
 ('spending', 0.9238353371620178),
 ('overcoming', 0.9234976768493652),
 ('behind-the-scenes', 0.9234638214111328),
 ('separating', 0.923354983329773),
 ('confronts', 0.923233151435852),
 ('prosecutor', 0.9231802821159363),
 ('shorten', 0.9226648807525635)]

In [25]:
# example - "the"
most_similar("the", word_embeddings)

[('theory', 0.9810282588005066),
 ('thein', 0.9804803133010864),
 ('theo', 0.9800756573677063),
 ('thematic', 0.9793440103530884),
 ('thee', 0.9792044162750244),
 ('theater', 0.9780211448669434),
 ('theta', 0.9779319167137146),
 ('thebes', 0.9778826236724854),
 ('theological', 0.9774019718170166),
 ('themes', 0.976779580116272)]

For Q2, I used the static token embeddings from Problem 1 to build word-level embeddings by tokenizing each word in the Glove vocabulary file and averaging the corresponding subword embeddings. Then, I implemented the most_similar() function using cosine similarity and ran it on the example words.

I spent around 3 hours on Problem 2